# Stage 37A: frozen pseudo-private benchmark manifest

500 training wells、既存62-cut design-validation、未開封120 confirmation wellsを再抽選せず固定します。Stage 37Aは予測も学習も行わず、confirmation TVTを読みません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'
if not (repo_dir/'.git').is_dir():
    subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else:
    subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
artifact_dir.mkdir(parents=True,exist_ok=True)
def run_checked(command):
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout,flush=True)
    if result.returncode:
        print(result.stderr,flush=True); raise RuntimeError(f'command failed: {command}')


## 既存splitを固定

Stage 24A v003の500/120 splitとStage 21Bの58-well design setを使います。trainingは500 wellsに属する全replay-eligible short-prefix cutsへ拡張されます。

In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
design_run=artifact_dir/'stage21b_prefix_confidence_full_v001'
required=[stage16b_run/'well_assignments.parquet',stage17a_run/'cut_report.parquet',design_run/'confidence_cut_report.parquet']
for path in required: assert path.is_file(),path
SPLIT_ID='stage37a_locked_split_manifest_v001'; split_run=artifact_dir/SPLIT_ID
if not ((split_run/'training_cut_ids.parquet').is_file() and (split_run/'confirmation_cut_ids.parquet').is_file()):
    if split_run.exists(): shutil.rmtree(split_run)
    run_checked(['uv','run','rogii-scaled-emission-manifest','--stage17a-run',str(stage17a_run),'--design-validation-run',str(design_run),'--artifact-dir',str(artifact_dir),'--run-id',SPLIT_ID,'--training-wells-per-fold','100','--confirmation-wells-per-fold','24','--target-fraction','0.30','--seed','42'])
split_summary=json.loads((split_run/'summary.json').read_text())
assert split_summary['training_wells']==500 and split_summary['confirmation_wells']==120,split_summary
RUN_ID='stage37a_pseudo_private_manifest_v001'; run_dir=artifact_dir/RUN_ID
if not (run_dir/'summary.json').is_file():
    if run_dir.exists(): shutil.rmtree(run_dir)
    run_checked(['uv','run','rogii-pseudo-private-manifest','--config','configs/experiment/stage37a_pseudo_private_manifest.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--split-run',str(split_run),'--design-validation-run',str(design_run),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID])
summary=json.loads((run_dir/'summary.json').read_text())
summary

In [ ]:
{key:summary[key] for key in ['stage37a_complete','promoted_to_stage37b_top_pf_replay','pseudo_private_manifest_sha256','roles','overlaps','confirmation_locked','confirmation_target_columns_read','gates','next_step']}

最後の辞書を共有してください。全gate通過後にだけStage 37Bでtraining/designのtop-PF proxyを計算します。confirmation 120 wellsは引き続き開封しません。